# 01 Project Overview and Score Release
## 프로젝트 개요와 공개용 점수 데이터

이 노트북은 AMEX 원자료를 다시 학습하지 않고, 공개 repo에 포함된 masked score sample과 aggregate output만으로 프로젝트 흐름을 확인한다.

핵심 질문은 단순 예측이 아니라, 제한된 manual review capacity에서 어떤 고객을 먼저 검토해야 하는가이다.


## Setup
### 설정

공개 sample, 결과표, 그림 저장 경로를 설정한다.


In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_DIR = DATA_DIR / "sample"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

def read_table(name):
    path = TABLE_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def save_figure(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    return path

def as_percent(series, digits=2):
    return (series.astype(float) * 100).round(digits)


## Public artifact inventory
### 공개 산출물 목록

GitHub에 포함된 파일이 어떤 분석 결과를 재현하는지 확인한다.


In [2]:
table_files = sorted(TABLE_DIR.glob("*.csv"))
figure_files = sorted(FIGURE_DIR.glob("*.png"))
sample_files = sorted(SAMPLE_DIR.glob("*.csv"))

inventory = pd.DataFrame(
    {
        "artifact_group": ["table"] * len(table_files) + ["figure"] * len(figure_files) + ["sample"] * len(sample_files),
        "file_name": [p.name for p in table_files + figure_files + sample_files],
        "size_kb": [round(p.stat().st_size / 1024, 1) for p in table_files + figure_files + sample_files],
    }
)
display(inventory)
display(inventory.groupby("artifact_group", as_index=False).agg(file_count=("file_name", "count"), total_kb=("size_kb", "sum")))


,artifact_group,file_name,size_kb
0,table,cost_scenario_summary.csv,10.6
1,table,monitoring_bucket_summary.csv,0.7
2,table,normalized_net_benefit_by_review_scope.csv,20.1
3,table,risk_decile_summary.csv,0.7
4,table,top10_to_top100_cumulative_summary.csv,0.6
5,table,topk_policy_tradeoff.csv,0.5
6,table,weighted_policy_tradeoff.csv,0.6
7,figure,default_rate_by_risk_band.png,31.7
8,figure,normalized_net_benefit_by_review_scope.png,240.8
9,figure,risk_bucket_decile_validation.png,100.0


,artifact_group,file_count,total_kb
0,figure,4,435.7
1,sample,1,130.1
2,table,7,33.8


공개 repo는 원본 customer-month 데이터 대신 masked sample과 aggregate 결과표를 포함한다. 따라서 노트북은 전체 모델을 재학습하기보다, 최종 decisioning layer의 결과를 재현하고 검증하는 역할을 한다.


## Masked score sample
### 마스킹된 점수 샘플

고객별 risk score, rank, risk band, review action 후보가 포함된 공개 sample을 확인한다.


In [3]:
score_sample = pd.read_csv(SAMPLE_DIR / "public_sample_scores.csv")
print(f"Sample shape: {score_sample.shape}")
display(score_sample.head(10))

band_summary = (
    score_sample.groupby("risk_band", as_index=False)
    .agg(
        customers=("masked_customer_id", "count"),
        defaults=("y_true", "sum"),
        avg_risk_score=("risk_score", "mean"),
        min_percentile=("risk_percentile", "min"),
        max_percentile=("risk_percentile", "max"),
    )
    .sort_values("avg_risk_score", ascending=False)
)
band_summary["default_rate_pct"] = (band_summary["defaults"] / band_summary["customers"] * 100).round(2)
display(band_summary)


Sample shape: (1000, 11)


,masked_customer_id,y_true,risk_score,risk_rank,risk_percentile,risk_band,review_priority,action_candidate,score_month,model_version,policy_version
0,0f75afc8...,1,0.999092,1,0.000218,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
1,3c1d5685...,1,0.999003,2,0.000436,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
2,1ea68239...,1,0.999000,3,0.000654,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
3,c5c6a560...,1,0.998997,4,0.000872,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
4,08233ac9...,1,0.998996,5,0.001090,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
5,84aedebe...,1,0.998995,6,0.001307,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
6,740642b7...,1,0.998995,7,0.001525,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
7,25e4f99c...,1,0.998988,8,0.001743,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
8,93753f53...,1,0.998980,9,0.001961,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1
9,a825ae8a...,1,0.998979,10,0.002179,Critical,1,Urgent manual review candidate,2018-03-31,best_equal_8models,policy_v1


,risk_band,customers,defaults,avg_risk_score,min_percentile,max_percentile,default_rate_pct
0,Critical,1000,1000,0.998541,0.000218,0.217906,100.0


공개 sample은 고객 식별자를 마스킹한 상태로 risk score의 운영 사용 방식을 보여준다. 높은 점수 고객은 `Critical` 또는 `High` risk band로 분류되어 manual review 우선순위가 높다.


## Original pipeline summary
### 원래 실험 흐름 요약

원본 Colab 노트북은 대용량 AMEX customer-month 데이터를 처리했지만, 공개 repo에서는 그 실행 로그를 그대로 올리지 않는다.


In [4]:
pipeline_summary = pd.DataFrame(
    [
        {
            "stage": "Feature engineering",
            "original_run": "Customer-month parquet -> customer-level feature table",
            "public_repo_role": "Summarized in docs and code modules",
        },
        {
            "stage": "Modeling",
            "original_run": "LightGBM, XGBoost, CatBoost, MLP, and blended OOF scores",
            "public_repo_role": "Final score sample and ranking outputs",
        },
        {
            "stage": "Decisioning",
            "original_run": "Top-K review policy, weighted correction, cost scenarios",
            "public_repo_role": "Fully reproduced from public aggregate CSVs",
        },
        {
            "stage": "Governance",
            "original_run": "Manual review queue and monitoring tables",
            "public_repo_role": "Documented as a decision-support prototype",
        },
    ]
)
display(pipeline_summary)


,stage,original_run,public_repo_role
0,Feature engineering,Customer-month parquet -> customer-level featu...,Summarized in docs and code modules
1,Modeling,"LightGBM, XGBoost, CatBoost, MLP, and blended ...",Final score sample and ranking outputs
2,Decisioning,"Top-K review policy, weighted correction, cost...",Fully reproduced from public aggregate CSVs
3,Governance,Manual review queue and monitoring tables,Documented as a decision-support prototype


포트폴리오에서 강조할 부분은 모델 leaderboard가 아니라, 예측 점수를 운영 의사결정 레이어로 바꾸는 과정이다. 따라서 이후 노트북은 ranking validation과 policy simulation에 집중한다.
